In [ ]:
# ═══ 1. התקנות (T4) ═══
import subprocess
subprocess.run(["pip", "install", "-q", "diffusers", "transformers", "accelerate", "safetensors", "sentencepiece"], check=False)
print("התקנות הושלמו")


In [ ]:
# ═══ 2. לאה קנונית: סטיל פתיחה עם הלורה (SDXL + lea-lora מ-Hugging Face) ═══
import torch
from diffusers import StableDiffusionXLPipeline
from huggingface_hub import hf_hub_download
DEV = "cuda"
pipe = StableDiffusionXLPipeline.from_pretrained("SG161222/RealVisXL_V5.0", torch_dtype=torch.float16, variant="fp16").to(DEV)
pipe.load_lora_weights(hf_hub_download("shimi-net/lea-lora", "lea_lora.safetensors"))
PROMPT = ("photorealistic full body photograph of Leah, olive tan skin with soft freckles, grey-green eyes, "
          "full dark brows, full pink lips, dark brown hair in a messy high bun with loose strands, "
          "huge natural heavy bust, soft round belly, hourglass figure, narrow waist, wide round hips, "
          "wearing a minimal micro bikini, holding a dance pole with one hand, "
          "inside a dim pole dance studio with purple neon light, facing camera, sharp focus, head to feet")
NEG = "deformed, distorted, extra limbs, bad anatomy, plastic skin, thin, flat chest, blurry, watermark, text, child"
g = torch.Generator(DEV).manual_seed(161101)
img = pipe(PROMPT, negative_prompt=NEG, width=832, height=1216,
           num_inference_steps=28, guidance_scale=5.5, generator=g).images[0]
img.save("/content/lea_pole_still.png")
print("סטיל פתיחה נוצר")
img.resize((208, 304))


In [ ]:
# ═══ 3. לופ: 5 שניות וידאו — LTX img2video מהסטיל של לאה ═══
import torch, gc
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video
del pipe
gc.collect()
torch.cuda.empty_cache()
repo = None
for cand in ["Lightricks/LTX-Video-2b", "a-rasi/LTX-Video-2b", "Lightricks/LTX-Video"]:
    try:
        vpipe = LTXImageToVideoPipeline.from_pretrained(cand, torch_dtype=torch.float16)
        repo = cand
        break
    except Exception as e:
        print("לא נמצא:", cand)
        vpipe = None
assert repo is not None, "לא נמצא מודל LTX — עצור ודווח לשימי פיתוח"
vpipe.enable_model_cpu_offload()
try:
    vpipe.vae.enable_tiling()
    vpipe.vae.enable_slicing()
except Exception:
    pass
NEG_V = "blurry, distorted face, deformed, extra limbs, morphing, low quality, watermark, text"
PROMPT_V = ("Leah dances gracefully around the dance pole, spinning and twirling around the pole, "
            "pole dance performance, smooth continuous motion, dynamic movement, camera fixed")
print("מייצר 121 פריימים (5 שניות)...")
out = vpipe(image=img, prompt=PROMPT_V, negative_prompt=NEG_V,
            width=480, height=704, num_frames=121,
            num_inference_steps=40, guidance_scale=3.0).frames[0]
export_to_video(out, "/content/lea_pole_5s.mp4", fps=24)
print("הוידאו נוצר: /content/lea_pole_5s.mp4")


In [ ]:
# ═══ 4. העלאה לקישור קבוע + הורדה ═══
import subprocess, shutil
r = subprocess.run(["curl", "-s", "-F", "reqtype=fileupload", "-F", "fileToUpload=@/content/lea_pole_5s.mp4",
                    "https://catbox.moe/user/api.php"], capture_output=True, text=True, timeout=300)
url = r.stdout.strip()
print("קישור קבוע:", url if url.startswith("http") else ("נכשל: " + url[:120]))
try:
    from google.colab import files
    files.download("/content/lea_pole_5s.mp4")
except Exception:
    pass
# הקישור גם נשמר כאן — העתק אותו לשימי פיתוח לאימות
with open("/content/LINK.txt", "w") as f:
    f.write(url)
